# Lekcija 10 - AI agenti v produkciji

V tej lekciji se boste naučili **produkcijskih vzorcev** za AI agente z uporabo Microsoft Agent Framework (Python). Obravnavali bomo:

- **Opazovanje** — dodajanje časovnih podatkov in beleženje interakcij agentov
- **Vrednotenje** — uporaba ocenjevalnega agenta za oceno kakovosti odgovorov
- **Upravljanje stroškov** — strategije optimizacije števila tokenov in izbire modela

Scenarij je **potovalni agent**, ki pomaga uporabnikom pri načrtovanju potovanj, s spremljanjem in vrednotenjem na vrhu.


## Nastavitev


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Premisleki za proizvodnjo

Premik AI agentov iz prototipov v proizvodnjo zahteva skrbno pozornost trem stebrom:

1. **Opazovanje** — Potrebujete vpogled v to, kaj agent dela, koliko časa potrebuje in katere orodja kliče. Brez sledenja in beleženja je odpravljanje napak v proizvodnji skoraj nemogoče.

2. **Vrednotenje** — Avtomatizirane kvalitativne kontrole zagotavljajo, da agentovi odgovori ostajajo natančni, popolni in koristni skozi čas. Agent za vrednotenje lahko ocenjuje odgovore glede na definirane kriterije.

3. **Upravljanje stroškov** — Poraba tokenov neposredno vpliva na stroške. Strategije kot so optimizacija poziva, izbor modela in medpomnjenje pomagajo obvladovati stroške brez žrtvovanja kakovosti.


## Gradnja opaznega agenta

Določimo orodja za potovanje in zavijemo klic agenta s časovnim merjenjem, da lahko spremljamo zakasnitev. V produkciji bi integrirali z OpenTelemetry ali podobnim sistemom za sledenje.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Vzorci ocenjevanja

Pogosta produkcijska praksa je uporaba drugega agent za **ocenjevalca**. Ocenjevalec oceni odgovor primarnega agenta glede na vnaprej določena merila, kot so popolnost, natančnost in koristnost.

To omogoča:
- Avtomatizirane kakovostne meje, preden odgovori dosežejo uporabnike
- Odkritje regresije, ko se spreminjajo pozivi ali modeli
- Stalno spremljanje uspešnosti agenta skozi čas


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategije upravljanja stroškov

Nadzor stroškov je ključnega pomena za proizvodne AI agente. Tukaj so ključne strategije:

| Strategija | Opis |
|---|---|
| **Optimizacija pozivov** | Ohranite sistemska navodila jedrnata. Odstranite odvečen kontekst za zmanjšanje vhodnih tokenov. |
| **Izbira modela** | Za preproste naloge, kot so klasifikacija ali ekstrakcija, uporabite manjše, cenejše modele (npr. GPT-4o-mini) in večje modele rezervirajte za kompleksno sklepanje. |
| **Predpomnjenje** | Predpomnite rezultate orodij in pogosta poizvedovanja, da se izognete podvojenim API klicem. |
| **Proračuni za tokene** | Nastavite omejitve `max_tokens`, da preprečite nepričakovano dolge odgovore. |
| **Združevanje** | Kjer je mogoče, združite več uporabniških poizvedb v en sam API klic. |

V praksi dobro deluje vrstni pristop: enostavne zahteve pošljite hitremu in poceni modelu, kompleksne pa le preusmerite na zmogljivejšega.


## Povzetek

V tej lekciji ste se naučili:

1. **Dodati opazljivost** interakcijam agentov z merjenjem časa in beleženjem, kar postavlja temelje za sledenje in spremljanje.
2. **Samodejno ocenjevati odzive agentov** z uporabo ocenjevalnega agenta, ki ocenjuje popolnost, točnost in uporabnost.
3. **Upravljati stroške** z optimizacijo pozivov, izbiro modela, predpomnjenjem in proračuni tokenov.

Ti proizvodni vzorci pomagajo zagotoviti, da so vaši AI agenti zanesljivi, merljivi in stroškovno učinkoviti v velikem obsegu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
